# Introdução

Neste notebook iremos construir um modelo de Análise de Sentimentos com [Pytorch](https://pytorch.org/) e [torchtext](https://docs.pytorch.org/text/stable/index.html). Para isso iremos usar um modelo de NLP chamado Neural Bag of Words (NBoW) e o dataset do [IMDB](https://huggingface.co/datasets/stanfordnlp/imdb) disponível no Hugging Face.

# Bibliotecas

Primeiro, vamos importar as bibliotecas necessárias. Usaremos o módulo `datasets` para obtermos o dataset dessa aplicação, `numpy` para computação numérica, `torch` para computação tensorial, `torch.nn` para redes neurais, `torch.optim` para usarmos alguns algoritmos de otimização nas nossas redes neurais, `nltk` para processamento de texto e `tqdm` para mostrar o progresso das operações.

In [31]:
import collections
import itertools

import datasets
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import nltk

Para manter a reprodutibilidade, vamos definir uma seed específica para as bibliotecas usarem na geração interna de números pseudo-aleatórios.

In [ ]:
seed = 42

np.random.seed(seed)
torch.manual_seed(seed)
torch.backends.cudnn.deterministic = True

# Carregando o Dataset

Nosso dataset é composto por $3$ conjuntos de dados: `train`, `test` e `unsupervised` e nessa atividade importa apenas os dois primeiros conjuntos de dados. Os dados de `train` serão usados para o treinamento do nosso modelo e os de `test` serão usados para avaliarmos o desempenho final do modelo.

Como podemos ver abaixo, ambos os conjuntos `train` e `test` possuem $25000$ amostras e duas features: o conteúdo textual da crítica do filme em `text` e a classificação da crítica como positiva ou negativa em `label`. A feature `text` é do tipo `Value("string")` e a feature `label` é do tipo `ClassLabel` com classes `neg` e `pos`.

In [33]:
dataset = datasets.load_dataset("imdb")

In [34]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [35]:
dataset["train"].features

{'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}

Agora, antes de prosseguirmos para o processamento do nosso dataset precisamos dividir ele em conjuntos de dados de treinamento, validação e teste:

- **Dados de Treinamento**: Os dados de treinamento serão usados para treinar nosso modelo e fazê-lo capturar os padrões escondidos no nosso dataset.
- **Dados de Validação**: Já os dados de validação serão usados para avaliarmos o desempenho do nosso modelo em dados não vistos, permitindo fazer um ajuste mais fino nos hiperparâmetros do nosso modelo.
- **Dados de Teste**: Por fim, os dados de teste serão usados para avaliar o desempenho real do nosso último e melhor modelo. O motivo de não usarmos os dados de teste no lugar dos dados de validação é pelo fato do nosso modelo poder sofrer com overfitting nos dados de teste e nos impedindo de avaliar seu desempenho real.

In [36]:
train_valid_split = dataset["train"].train_test_split(test_size=0.2, seed=seed)

train_data = train_valid_split["train"]
valid_data = train_valid_split["test"]
test_data = dataset["test"]

Abaixo podemos ver as features e a quantidade de amostras de cada conjunto de dados:

In [37]:
train_data, valid_data, test_data

(Dataset({
     features: ['text', 'label'],
     num_rows: 20000
 }),
 Dataset({
     features: ['text', 'label'],
     num_rows: 5000
 }),
 Dataset({
     features: ['text', 'label'],
     num_rows: 25000
 }))

Para termos uma ideia melhor, podemos imprimir a primeira amostra do nosso conjunto de dados de treinamento.

In [38]:
train_data[0]

{'text': 'Stage adaptations often have a major fault. They often come out looking like a film camera was simply placed on the stage (Such as "Night Mother"). Sidney Lumet\'s direction keeps the film alive, which is especially difficult since the picture offered him no real challenge. Still, it\'s nice to look at for what it is. The chemistry between Michael Caine and Christopher Reeve is quite brilliant. The dynamics of their relationship are surprising. Caine is fantastic as always, and Reeve gets one of his few chances to really act.<br /><br />I confess that I\'ve never seen Ira Levin\'s play, but I hear that Jay Presson Allen\'s adaptation is faithful. The script is incredibly convoluted, and keeps you guessing. "Deathtrap" is an enormously entertaining film, and is recommended for nearly all fans of stage and screen.<br /><br />7.4 out of 10',
 'label': 1}

# Tokenização

Antes de prosseguirmos, precisamos tokenizar o texto das nossas amostras, pois os algoritmos de machine learning, em sua maioria, são algoritmos numéricos e não trabalham com strings puras. Portanto, iremos dividir nossos textos em trechos menores e indivisíveis (os tokens) e convertê-los em números. A esse processo damos o nome de Tokenização.

Para isso vamos usar o `nltk.tokenize.word_tokenize`. Primeiro verificamos se o pacote `punkt` está disponível e caso contrário nós o instalamos:

In [39]:
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt")

try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    nltk.download("punkt_tab")

Agora, vamos criar uma função para tokenizar nosso texto usando o `nltk.tokenize.word_tokenize` e note que estamos considerando apenas os tokens que são palavras ou números, ignorando os tokens relacionados a pontuações e afins:

In [40]:
def tokenizer(text):
    tokens = nltk.tokenize.word_tokenize(text)
    return [token.lower() for token in tokens if token.isalnum()]

tokenizer("I don't know what I am doing, but I am doing!!!")

['i', 'do', 'know', 'what', 'i', 'am', 'doing', 'but', 'i', 'am', 'doing']

Precisamos criar uma nova feature `tokens` no nosso dataset que conterá os tokens de `text` em cada amostra. Para isso vamos criar uma função que será usada para mapear os tokens em cada amostra:

In [41]:
def tokenize_sample(sample, tokenizer, max_length):
    tokens = tokenizer(sample["text"])[:max_length]
    return {"tokens": tokens}

Com a função `tokenize_sample` podemos mapear os tokens de `text` em cada amostra do nosso dataset para a nova feature `tokens` usando o método `map` presente no tipo `Dataset` do nosso dataset e limitando a quantidade máxima de tokens em cada amostra para $256$:

In [42]:
max_length = 256

train_data = train_data.map(tokenize_sample, fn_kwargs={"tokenizer": tokenizer, "max_length": max_length})
test_data = test_data.map(tokenize_sample, fn_kwargs={"tokenizer": tokenizer, "max_length": max_length})

Agora podemos ver que nosso dataset tem uma nova feature chamada `tokens`:

In [43]:
train_data

Dataset({
    features: ['text', 'label', 'tokens'],
    num_rows: 20000
})

Analisando com mais detalhe podemos ver que essa feature é uma `List` com valores do tipo `Value` de `string`, ou seja, uma lista de strings (os tokens):

In [44]:
train_data.features

{'text': Value('string'),
 'label': ClassLabel(names=['neg', 'pos']),
 'tokens': List(Value('string'))}

Podemos verificar também se a tokenização realmente funcionou imprimindo parte do texto e dos tokens de uma amostra qualquer:

In [45]:
some_sample = train_data[0]

print(f"Text: {some_sample['text'][:43]}")
print(f"Tokens: {some_sample['tokens'][:7]}")

Text: Stage adaptations often have a major fault.
Tokens: ['stage', 'adaptations', 'often', 'have', 'a', 'major', 'fault']


# Criando o Vocabulário

Para usarmos o modelo Neural Bag of Words precisamos codificar os tokens existentes no nosso dataset em números inteiros.

Para isso iremos criar o vocabulário do nosso dataset, ou seja, uma lista com os tokens existentes no nosso dataset, sendo cada um associado a um inteiro único. Usaremos o índice da posição do token na lista de números como representação numérica para o token.

Para criarmos o vocabulário do nosso dataset iremos contar a frequência de cada token no dataset e filtraremos os tokens com uma frequência mínima `min_freq` e associaremos o índice do token na lista ao respectivo token.

In [46]:
def build_vocab(tokens, min_freq, special_tokens):
    counter = collections.Counter(itertools.chain.from_iterable(tokens))
    counter = {
        word: freq for word, freq in counter.items() if freq >= min_freq
    }

    for token in special_tokens:
        counter[token] = min_freq

    vocab = {
        word: i for i, (word, _) in enumerate(counter.items())
    }

    return vocab

Com essa função podemos criar nosso vocabulário. Note que a frequência mínima que estamos usando é 5 e note também que adicionamos ao vocabulário dois tokens chamados `"<unk>"` e `"<pad>"`. O token `"<unk>"` é usado no lugar de tokens que não existem no vocabulário e `"<pad>"` é usado para preencher frases (explicaremos isso mais adiante).

In [47]:
min_freq = 5
special_tokens = ["<unk>", "<pad>"]

vocab = build_vocab(
    tokens=train_data["tokens"],
    min_freq=min_freq,
    special_tokens=special_tokens
)

Como podemos ver nosso vocabulário tem $22064$ tokens:

In [48]:
len(vocab)

22064

Antes de prosseguirmos com a construção do nosso modelo, precisamos decidir o dispositivo que iremos usar para o treinamento do modelo, ou seja, a `CPU` ou a `GPU`.

In [49]:
device = torch.device("cpu")

# Carregando o Dataset no Pytorch

Para que o nosso dataset possa ser consumido pelo modelo que iremos treinar com o Pytorch precisamos criar uma nova classe `IMDBDataset` que herda de ``torch.utils.data.Dataset`. Essa classe vai permitir que o Pytorch tenha acesso aos dados de treinamento, bem como o vocabulário e o vetor que representa numericamente a sequência de tokens da crítica.

Além disso, o treinamento do nosso modelo acontecerá em lotes (batches) de $64$ frases (amostras do dataset), ou seja, nosso modelo irá treinar de maneira paralela com um lote inteiro de 64 críticas com o objetivo de otimizar o treinamento. Para isso definimos a função `collate_fn` que normaliza as amostras de um batch adicionando padding (o token `"<pad>"`) no final das amostras menores que a maior amostra do batch.

In [50]:
class IMDBDataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, vocab):
        self.data = hf_dataset
        self.vocab = vocab

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        text = self.data[index]["text"]
        label = float(self.data[index]["label"])

        tokens = tokenizer(text)
        numericalized_text = [self.vocab.get(token, self.vocab["<unk>"]) for token in tokens]

        return torch.tensor(numericalized_text, dtype=torch.long), torch.tensor(label, dtype=torch.float)

def collate_fn(batch):
    texts, labels = zip(*batch)

    texts_padded = nn.utils.rnn.pad_sequence(texts, batch_first=True, padding_value=vocab["<pad>"])
    labels = torch.stack(labels)

    return texts_padded.to(device), labels.to(device)

Com a classe `IMDBDataset` e função `collate_fn` definidas podemos criar os datasets e data loaders necessários para o Pytorch. Os data loaders da classe `torch.utils.data.DataLoader` dividem nosso dataset nos batches de tamanho $64$ de maneira aleatória e usando a nossa função `collate_fn`.

In [51]:
batch_size = 64

train_dataset = IMDBDataset(train_data, vocab)
valid_dataset = IMDBDataset(valid_data, vocab)
test_dataset = IMDBDataset(test_data, vocab)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

# Criando o Modelo Neural Bag of Words (NBoW)

Para criarmos o nosso modelo precisamos definir a classe `NBoWModel` que herda de `nn.Module`. Definimos essa classe com uma camada de embedding `nn.EmbeddingBag` e uma camada `nn.Linear`. A camada `nn.EmbeddingBag` é responsável por buscar os embeddings dos tokens da crítica e mesclá-los em um único embedding e a última camada é responsável por gerar um número a partir desse embedding, ou seja, o logit que posteriormente se tornará a probabilidade da crítica.

In [52]:
class NBoWModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, output_dim):
        super().__init__()

        self.embedding = nn.EmbeddingBag(vocab_size, embedding_dim, padding_idx=vocab["<pad>"])
        self.fc = nn.Linear(embedding_dim, output_dim)

    def forward(self, text):
        embedded = self.embedding(text)

        return self.fc(embedded)

Com a classe `NBoWModel` definida podemos criar nosso modelo, o otimizador e a função de perda do nosso modelo. Aqui, nosso modelo gerá embeddings de dimensão $64$ e dimensão de saída igual a $1$. O otimizador é o algoritmo de otimização que será usada para minizar a função de perda, ou seja, ajustar os parâmetros do modelo de modo a minimizar a função de perda. Já a função de perda é a métrica que diz quão ruim é nosso modelo.

In [53]:
model = NBoWModel(len(vocab), embedding_dim=64, output_dim=1).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss()

# Treinamento do Modelo

Para efetuarmos o treinamento do nosso modelo precisamos da função `train_epoch` que faz o treinamento do modelo em uma época. Para avaliarmos o desempenho do modelo usamos a função `evaluate_epoch` que calcula a acurácia, precisão e recall do modelo nos dados de validação. Dessa forma podemos acompanhar a evolução do treinamento do modelo e como as métricas de desempenho do modelo mudam com o passar das épocas.

In [54]:
def calculate_accuracy(predictions, y):
    rounded_predictions = torch.round(torch.sigmoid(predictions))

    correct = (rounded_predictions == y).float()
    acc = correct.sum() / len(correct)

    return acc

def calculate_precision_recall(predictions, y):
    rounded_preds = torch.round(torch.sigmoid(predictions))

    true_positives = ((rounded_preds == 1) & (y == 1)).float().sum()
    false_positives = ((rounded_preds == 1) & (y == 0)).float().sum()
    false_negatives = ((rounded_preds == 0) & (y == 1)).float().sum()

    epsilon = 1e-7
    precision = true_positives / (true_positives + false_positives + epsilon)
    recall = true_positives / (true_positives + false_negatives + epsilon)

    return precision, recall

def calculate_f1_score(precision, recall):
    return 2 * precision*recall / (precision + recall)

def train_epoch(model, iterator, optimizer, criterion):
    model.train()

    epoch_loss = 0
    epoch_accuracy = 0
    epoch_precision = 0
    epoch_recall = 0
    epoch_f1_score = 0

    for texts, labels in iterator:
        optimizer.zero_grad()

        predictions = model(texts).squeeze(1)
        loss = criterion(predictions, labels)

        accuracy = calculate_accuracy(predictions, labels)
        precision, recall = calculate_precision_recall(predictions, labels)
        f1_score = calculate_f1_score(precision, recall)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        epoch_accuracy += accuracy.item()
        epoch_precision += precision.item()
        epoch_recall += recall.item()
        epoch_f1_score += f1_score.item()

    epoch_loss /= len(iterator)
    epoch_accuracy /= len(iterator)
    epoch_precision /= len(iterator)
    epoch_recall /= len(iterator)
    epoch_f1_score /= len(iterator)

    return epoch_loss, epoch_accuracy, epoch_precision, epoch_recall, epoch_f1_score

def evaluate_epoch(model, iterator, criterion):
    model.eval()

    epoch_loss = 0
    epoch_accuracy = 0
    epoch_precision = 0
    epoch_recall = 0
    epoch_f1_score = 0

    with torch.no_grad():
        for texts, labels in iterator:
            predictions = model(texts).squeeze(1)
            loss = criterion(predictions, labels)

            accuracy = calculate_accuracy(predictions, labels)
            precision, recall = calculate_precision_recall(predictions, labels)
            f1_score = calculate_f1_score(precision, recall)

            epoch_loss += loss.item()
            epoch_accuracy += accuracy.item()
            epoch_precision += precision.item()
            epoch_recall += recall.item()
            epoch_f1_score += f1_score.item()

    epoch_loss /= len(iterator)
    epoch_accuracy /= len(iterator)
    epoch_precision /= len(iterator)
    epoch_recall /= len(iterator)
    epoch_f1_score /= len(iterator)

    return epoch_loss, epoch_accuracy, epoch_precision, epoch_recall, epoch_f1_score

Por fim, fazemos o treinamento do nosso modelo em $3$ épocas e usando o conjunto de dados de validação para avaliarmos o nosso modelo. Além disso, salvamos os pesos do melhor modelo encontrado no arquivo `modelo_nbow.pt`.

In [ ]:
EPOCHS = 3
best_valid_loss = float("inf")

for epoch in range(EPOCHS):
    train_loss, train_accuracy, train_precision, train_recall, train_f1_score = train_epoch(model, train_loader, optimizer, criterion)
    valid_loss, valid_accuracy, valid_precision, valid_recall, valid_f1_score = evaluate_epoch(model, valid_loader, criterion)

    print(f"Epoch: {epoch + 1}")
    print(f"\tTrain Loss: {train_loss:.3f} | Train Accuracy: {100*train_accuracy:.2f}% Train Precision: {100*train_precision:.2f}% Train Recall: {100*train_recall:.2f}% Train F1-Score: {100*train_f1_score:.2f}%")
    print(f"\tValid Loss: {valid_loss:.3f} | Valid Accuracy: {100*valid_accuracy:.2f}% Valid Precision: {100*valid_precision:.2f}% Valid Recall: {100*valid_recall:.2f}% Valid F1-Score: {100*valid_f1_score:.2f}%")

Epoch: 1
	Train Loss: 0.666 | Train Accuracy: 66.44% Train Precision: 67.00% Train Recall: 71.67% Train F1-Score: 67.99%
	Valid Loss: 0.618 | Valid Accuracy: 75.71% Valid Precision: 78.09% Valid Recall: 71.84% Valid F1-Score: 74.45%
Epoch: 2
	Train Loss: 0.541 | Train Accuracy: 80.03% Train Precision: 79.58% Train Recall: 80.53% Train F1-Score: 79.79%
	Valid Loss: 0.470 | Valid Accuracy: 83.23% Valid Precision: 84.48% Valid Recall: 81.72% Valid F1-Score: 82.85%
Epoch: 3
	Train Loss: 0.409 | Train Accuracy: 85.21% Train Precision: 84.72% Train Recall: 85.68% Train F1-Score: 85.01%
	Valid Loss: 0.378 | Valid Accuracy: 85.78% Valid Precision: 87.20% Valid Recall: 83.97% Valid F1-Score: 85.33%


Para avaliarmos a qualidade do nosso modelo final, iremos usar o conjunto de teste para analisarmos suas métricas de desempenho. Como podemos ver abaixo, todas as 3 métricas calculadas estão em torno de $85%$, um ótimo desempenho para um modelo como o `NBoW`.

In [56]:
test_loss, test_accuracy, test_precision, test_recall, test_f1_score = evaluate_epoch(model, test_loader, criterion)

print(f"Test Loss: {test_loss:.3f} | Test Accuracy: {100*test_accuracy:.2f}% Test Precision: {100*test_precision:.2f}% Test Recall: {100*test_recall:.2f}% Valid F1-Score: {100*test_f1_score:.2f}%")

Test Loss: 0.397 | Test Accuracy: 84.25% Test Precision: 86.50% Test Recall: 81.31% Valid F1-Score: 83.56%


# Predição

Para efetuarmos a predição do sentimento de uma crítica, precisamos apenas gerar os indices dos tokens da crítica e passar isso para o modelo. O modelo então vai calcular a predição (o logit) para esse conjunto de índices e usaremos a função sigmoid para converter isso em uma probabilidade. Com essa probabilidade podemos calcular o sentimento da nossa crítica, quanto mais próximo do $0$ mais negativa e quanto mais próximo de $1$ mais positiva é a crítica.

In [57]:
def predict_sentiment(model, text, vocab, device):
    model.eval()

    tokens = tokenizer(text)
    indices = [vocab.get(word, vocab["<unk>"]) for word in tokens]

    if len(indices) == 0:
        return "Texto inválido", 0.5

    tensor = torch.LongTensor(indices).to(device)
    tensor = tensor.unsqueeze(0)

    with torch.no_grad():
        raw_predict = model(tensor).squeeze(1)
        prob = torch.sigmoid(raw_predict).item()

    sentiment = "Positivo" if prob >= 0.5 else "Negativo"

    return sentiment, prob

Podemos testar nosso modelo com alguma frases e constatarmos que ele realmente funciona.

In [58]:
texts = [
    "This movie is so terrible",
    "I really like the movie, it is awesome",
    "I like the acting, but the history is not so good",
    "The photography is not good, but i loved the film direction and the soundtrack"
]

for text in texts:
    sentiment, probability = predict_sentiment(model, text, vocab, device)
    print(f"\"{text}\": {sentiment}, {probability:.2f}")

"This movie is so terrible": Negativo, 0.00
"I really like the movie, it is awesome": Positivo, 0.79
"I like the acting, but the history is not so good": Negativo, 0.13
"The photography is not good, but i loved the film direction and the soundtrack": Positivo, 0.92


Por curiosidade, abaixo temos a quantidade de parâmetros desse nosso pequeno modelo, apenas $1412161$ parâmetros.

In [59]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total de parâmetros: {total_params}")

Total de parâmetros: 1412161


# Exportando o Modelo e o Vocabulário

Para colocarmos nosso modelo em produção precisamos exportá-lo juntamente com o vocabulário do nosso dataset. Para isso devemos criar um arquivo com os pesos dos parâmetros da rede neural que treinamos para que a aplicação web possa usar esses pesos e fazer as inferências na aplicação. Além disso, precisamos do vocabulário para traduzirmos o texto da crítica em seus respectivos tokens.

Abaixo começamos exportando o arquivo `nbow_model.pt` que contém os pesos da nossa rede neural já treinada:

In [61]:
torch.save(model.state_dict(), "../application/nbow_model.pt")

Agora vamos crar e exportar o vocabulário do nosso dataset:

In [62]:
import json
from pathlib import Path

with open("../application/vocab.json", 'w', encoding='utf-8') as file:
    json.dump(vocab, file, ensure_ascii=False, indent=4)

Com isso feito, estamos prontos para darmos início à criação da nossa aplicação.